# Machine Learning Modeling — Advanced Reference
> **Level:** Advanced | **Goal:** End-to-end ML pipelines, evaluation, tuning, and interpretability

## Table of Contents
1. [Scikit-learn Pipelines](#pipelines)
2. [Feature Engineering & Selection](#features)
3. [Cross-Validation Strategies](#cv)
4. [Hyperparameter Tuning](#tuning)
5. [Model Evaluation — Full Metrics Suite](#evaluation)
6. [Ensemble Methods](#ensembles)
7. [Model Interpretability (SHAP)](#shap)
8. [Class Imbalance](#imbalance)
9. [Gradient Boosting (XGBoost / LightGBM)](#boosting)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

from sklearn.datasets import make_classification, make_regression, load_breast_cancer
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold, KFold,
    GridSearchCV, RandomizedSearchCV, learning_curve
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    OneHotEncoder, OrdinalEncoder, LabelEncoder,
    PolynomialFeatures, PowerTransformer
)
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.feature_selection import (
    SelectKBest, f_classif, mutual_info_classif,
    RFE, SelectFromModel, RFECV
)
from sklearn.linear_model import LogisticRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor,
    GradientBoostingClassifier, AdaBoostClassifier,
    VotingClassifier, StackingClassifier, BaggingClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, log_loss,
    confusion_matrix, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
    mean_squared_error, mean_absolute_error, r2_score
)
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay

rng = np.random.default_rng(42)
print("Setup complete")

---
## 1 · Scikit-learn Pipelines <a id='pipelines'></a>

Pipelines chain preprocessing + model into one object:
- **No data leakage** — transformers are fit only on training folds
- **Single `.fit()` / `.predict()` interface**
- **Hyperparameter tuning across all steps simultaneously**

In [ ]:
# ── Build a realistic dataset with mixed types ─────────────────
n = 2000
df = pd.DataFrame({
    'age':        rng.integers(18, 70, n).astype(float),
    'income':     rng.lognormal(10, 1, n),
    'debt_ratio': rng.beta(2, 5, n),
    'n_products':  rng.integers(0, 10, n).astype(float),
    'region':     rng.choice(['North','South','East','West'], n),
    'job_type':   rng.choice(['employed','self_employed','retired','student'], n),
})

# Inject missing values
for col, rate in [('age', 0.05), ('income', 0.08), ('region', 0.03)]:
    mask = rng.random(n) < rate
    df.loc[mask, col] = np.nan

# Target: default (binary)
logit = (-3 + 0.02*df['age'].fillna(40) - 0.3*df['income'].fillna(df['income'].median())/10000
         + 3*df['debt_ratio'] + rng.normal(0, 0.5, n))
y = (1 / (1 + np.exp(-logit)) > 0.5).astype(int)

X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.2, stratify=y, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Class balance: {y_train.mean():.2%} positive")

In [ ]:
# ── ColumnTransformer + Pipeline ──────────────────────────────
numeric_features  = ['age', 'income', 'debt_ratio', 'n_products']
categorical_features = ['region', 'job_type']

numeric_transformer = Pipeline([
    ('imputer', KNNImputer(n_neighbors=5)),
    ('scaler',  RobustScaler()),          # robust to outliers
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer,  numeric_features),
    ('cat', categorical_transformer, categorical_features),
], remainder='drop')

pipeline = Pipeline([
    ('prep',   preprocessor),
    ('model',  LogisticRegression(max_iter=1000, class_weight='balanced'))
])

pipeline.fit(X_train, y_train)
y_pred  = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, y_proba):.4f}")
print("\n", classification_report(y_test, y_pred))

---
## 2 · Feature Engineering & Selection <a id='features'></a>

In [ ]:
# ── Interaction features with PolynomialFeatures ───────────────
from sklearn.pipeline import Pipeline

poly_pipeline = Pipeline([
    ('prep',   preprocessor),
    ('poly',   PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
    ('select', SelectKBest(mutual_info_classif, k=20)),
    ('model',  LogisticRegression(max_iter=1000, class_weight='balanced'))
])

cv_scores = cross_val_score(poly_pipeline, X_train, y_train,
                            cv=StratifiedKFold(5), scoring='roc_auc')
print(f"Poly+SelectKBest CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

In [ ]:
# ── Feature importance from tree models ───────────────────────
rf_pipeline = Pipeline([
    ('prep',  preprocessor),
    ('model', RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42))
])
rf_pipeline.fit(X_train, y_train)

# Get feature names from ColumnTransformer
cat_names = rf_pipeline.named_steps['prep'].named_transformers_['cat']['encoder'].get_feature_names_out(categorical_features)
feature_names = numeric_features + list(cat_names)
importances = rf_pipeline.named_steps['model'].feature_importances_

feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
feat_imp.head(12).plot.barh(ax=ax, color='steelblue')
ax.invert_yaxis()
ax.set_title('Top 12 Feature Importances (Random Forest)', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3 · Cross-Validation Strategies <a id='cv'></a>

| Strategy | When to use |
|---|---|
| `KFold(k=5)` | Regression, balanced classes |
| `StratifiedKFold(k=5)` | Classification — preserves class ratio |
| `TimeSeriesSplit` | Time series — respect temporal order |
| `GroupKFold` | Groups must not span folds (user-level data) |
| `RepeatedStratifiedKFold` | Small datasets — reduce variance of estimate |

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, GroupKFold, RepeatedStratifiedKFold

# ── Compare CV strategies ──────────────────────────────────────
strategies = {
    'KFold-5':                 KFold(n_splits=5, shuffle=True, random_state=42),
    'StratifiedKFold-5':       StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    'RepeatedStratifiedKFold': RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42),
}

# Preprocess once for speed
X_preprocessed = preprocessor.fit_transform(X_train, y_train)

model = LogisticRegression(max_iter=1000, class_weight='balanced')
for name, cv in strategies.items():
    scores = cross_val_score(model, X_preprocessed, y_train, cv=cv, scoring='roc_auc')
    print(f"{name:35s}: {scores.mean():.4f} ± {scores.std():.4f}  (n_evals={len(scores)})")

In [ ]:
# ── Learning curves: detect bias/variance ─────────────────────
train_sizes, train_scores, val_scores = learning_curve(
    rf_pipeline, X_train, y_train,
    cv=StratifiedKFold(5),
    scoring='roc_auc',
    train_sizes=np.linspace(0.1, 1.0, 10),
    n_jobs=-1
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, train_scores.mean(1), 'o-', label='Train', color='steelblue')
ax.fill_between(train_sizes, train_scores.mean(1)-train_scores.std(1),
                train_scores.mean(1)+train_scores.std(1), alpha=0.15, color='steelblue')
ax.plot(train_sizes, val_scores.mean(1), 'o-', label='Validation', color='crimson')
ax.fill_between(train_sizes, val_scores.mean(1)-val_scores.std(1),
                val_scores.mean(1)+val_scores.std(1), alpha=0.15, color='crimson')
ax.set(title='Learning Curves (Random Forest)', xlabel='Training size',
       ylabel='ROC-AUC')
ax.legend()
ax.spines[['top','right']].set_visible(False)
plt.show()

---
## 4 · Hyperparameter Tuning <a id='tuning'></a>

In [ ]:
from scipy.stats import randint, uniform

# ── RandomizedSearchCV (faster than Grid for large spaces) ────
param_dist = {
    'model__n_estimators':     randint(100, 500),
    'model__max_depth':        [None, 5, 10, 15, 20],
    'model__min_samples_leaf': randint(1, 20),
    'model__max_features':     ['sqrt', 'log2', 0.5],
    'model__class_weight':     ['balanced', 'balanced_subsample']
}

search = RandomizedSearchCV(
    rf_pipeline,
    param_distributions=param_dist,
    n_iter=20,
    cv=StratifiedKFold(3),
    scoring='roc_auc',
    n_jobs=-1,
    refit=True,
    random_state=42,
    verbose=0
)
search.fit(X_train, y_train)

print(f"Best CV ROC-AUC: {search.best_score_:.4f}")
print(f"Best params: {search.best_params_}")

best_model = search.best_estimator_
y_pred_best  = best_model.predict(X_test)
y_proba_best = best_model.predict_proba(X_test)[:, 1]
print(f"\nTest ROC-AUC: {roc_auc_score(y_test, y_proba_best):.4f}")

---
## 5 · Model Evaluation — Full Metrics Suite <a id='evaluation'></a>

In [ ]:
# ── Comprehensive evaluation function ─────────────────────────
def evaluate_classifier(model, X_test, y_test, model_name='Model', threshold=0.5):
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred  = (y_proba >= threshold).astype(int)

    metrics = {
        'Accuracy':        accuracy_score(y_test, y_pred),
        'Precision':       precision_score(y_test, y_pred),
        'Recall':          recall_score(y_test, y_pred),
        'F1':              f1_score(y_test, y_pred),
        'ROC-AUC':         roc_auc_score(y_test, y_proba),
        'PR-AUC':          average_precision_score(y_test, y_proba),
        'Log-Loss':        log_loss(y_test, y_proba),
    }
    print(f"── {model_name} ──")
    for name, val in metrics.items():
        print(f"  {name:15s}: {val:.4f}")
    return metrics

evaluate_classifier(best_model, X_test, y_test, 'Random Forest (Tuned)')

In [ ]:
# ── Visual evaluation: 4-panel diagnostic ─────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# 1. Confusion Matrix
ConfusionMatrixDisplay.from_estimator(best_model, X_test, y_test,
                                       normalize='true', cmap='Blues', ax=axes[0])
axes[0].set_title('Confusion Matrix (normalized)')

# 2. ROC Curve
RocCurveDisplay.from_estimator(best_model, X_test, y_test, ax=axes[1])
axes[1].set_title('ROC Curve')

# 3. Precision-Recall Curve
PrecisionRecallDisplay.from_estimator(best_model, X_test, y_test, ax=axes[2])
axes[2].set_title('Precision-Recall Curve')

# 4. Calibration Plot
CalibrationDisplay.from_estimator(best_model, X_test, y_test,
                                   n_bins=10, ax=axes[3])
axes[3].set_title('Calibration Curve')

plt.suptitle('Model Evaluation Dashboard', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6 · Ensemble Methods <a id='ensembles'></a>

In [ ]:
# ── Stacking Classifier ────────────────────────────────────────
base_estimators = [
    ('lr',  LogisticRegression(max_iter=1000, class_weight='balanced')),
    ('rf',  RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)),
    ('svm', SVC(probability=True, kernel='rbf', class_weight='balanced', random_state=42)),
]
meta_learner = LogisticRegression(max_iter=500)

stacking_pipeline = Pipeline([
    ('prep', preprocessor),
    ('stack', StackingClassifier(
        estimators=base_estimators,
        final_estimator=meta_learner,
        cv=StratifiedKFold(5),
        passthrough=False,
        stack_method='predict_proba'
    ))
])

stacking_pipeline.fit(X_train, y_train)
y_proba_stack = stacking_pipeline.predict_proba(X_test)[:, 1]
print(f"Stacking ROC-AUC: {roc_auc_score(y_test, y_proba_stack):.4f}")

---
## 7 · Model Interpretability (SHAP) <a id='shap'></a>

SHAP (SHapley Additive exPlanations) decomposes each prediction into feature contributions.

- `TreeExplainer` — fast for tree-based models
- `LinearExplainer` — for linear models
- `KernelExplainer` — model-agnostic (slow)

In [ ]:
try:
    import shap

    # Extract fitted RF from pipeline
    rf_model = best_model.named_steps['model']
    X_test_transformed = best_model.named_steps['prep'].transform(X_test)

    explainer = shap.TreeExplainer(rf_model)
    shap_values = explainer.shap_values(X_test_transformed)

    # Global importance
    shap.summary_plot(shap_values[1], X_test_transformed,
                      feature_names=feature_names, show=True)

    # Single prediction explanation
    idx = 0
    shap.waterfall_plot(shap.Explanation(
        values=shap_values[1][idx],
        base_values=explainer.expected_value[1],
        data=X_test_transformed[idx],
        feature_names=feature_names
    ))

except ImportError:
    print("Install shap: pip install shap")
    print("SHAP provides global and local model explanations.")
    print("Key plots:")
    print("  shap.summary_plot()   — global feature importance beeswarm")
    print("  shap.waterfall_plot() — single prediction explanation")
    print("  shap.dependence_plot()— feature interaction with another")

---
## 8 · Class Imbalance <a id='imbalance'></a>

In [ ]:
# ── Strategies for imbalanced data ────────────────────────────
print("Strategies for class imbalance:")
print("  1. class_weight='balanced' in sklearn models")
print("  2. SMOTE/ADASYN (oversample minority class) — via imbalanced-learn")
print("  3. Undersampling majority class")
print("  4. Adjust decision threshold")
print("  5. Use PR-AUC / F1 metrics instead of accuracy")

# ── Threshold optimization (F1 maximization) ─────────────────
from sklearn.metrics import precision_recall_curve

y_proba_rf = best_model.predict_proba(X_test)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test, y_proba_rf)
f1_scores = 2 * precision * recall / (precision + recall + 1e-9)
best_idx = f1_scores.argmax()
best_threshold = thresholds[best_idx]

print(f"\nDefault threshold (0.5):")
print(classification_report(y_test, (y_proba_rf >= 0.5).astype(int)))

print(f"Optimal threshold ({best_threshold:.3f}):")
print(classification_report(y_test, (y_proba_rf >= best_threshold).astype(int)))

---
## 9 · Gradient Boosting: XGBoost / LightGBM <a id='boosting'></a>

### Key hyperparameters

| Parameter | Effect | Typical range |
|---|---|---|
| `n_estimators` | # of trees | 100–1000 (use early stopping) |
| `learning_rate` | Step size | 0.01–0.3 (lower = more trees needed) |
| `max_depth` | Tree depth | 3–8 |
| `subsample` | Row sampling | 0.6–1.0 |
| `colsample_bytree` | Column sampling | 0.6–1.0 |
| `min_child_weight` | Regularization | 1–10 |
| `reg_alpha` / `reg_lambda` | L1/L2 reg | 0–10 |

In [ ]:
try:
    import xgboost as xgb
    import lightgbm as lgb

    X_tr_prep = preprocessor.fit_transform(X_train, y_train)
    X_te_prep = preprocessor.transform(X_test)

    # ── XGBoost with early stopping ────────────────────────────
    xgb_model = xgb.XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=(y_train==0).sum() / (y_train==1).sum(),  # handle imbalance
        eval_metric='auc',
        early_stopping_rounds=20,
        random_state=42
    )
    X_trn, X_val, y_trn, y_val = train_test_split(X_tr_prep, y_train, test_size=0.15, random_state=0)
    xgb_model.fit(X_trn, y_trn, eval_set=[(X_val, y_val)], verbose=0)

    y_proba_xgb = xgb_model.predict_proba(X_te_prep)[:, 1]
    print(f"XGBoost ROC-AUC: {roc_auc_score(y_test, y_proba_xgb):.4f}  (best iter={xgb_model.best_iteration})")

    # ── LightGBM ───────────────────────────────────────────────
    lgb_model = lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        is_unbalance=True,
        random_state=42,
        verbose=-1
    )
    lgb_model.fit(X_trn, y_trn,
                  eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(20, verbose=False)])

    y_proba_lgb = lgb_model.predict_proba(X_te_prep)[:, 1]
    print(f"LightGBM ROC-AUC: {roc_auc_score(y_test, y_proba_lgb):.4f}")

except ImportError as e:
    print(f"Install: pip install xgboost lightgbm\nError: {e}")
    print("\nXGBoost/LightGBM are go-to boosting libraries:")
    print("  - Faster than sklearn GradientBoosting")
    print("  - Native support for early stopping & missing values")
    print("  - LightGBM is faster on large datasets (leaf-wise vs depth-wise)")